# The mission — how it works

This is the explanation and reference for `quicksat.utils.mission.Mission`. It covers the one mission that every budget shares, what it derives, and what it does not model.

It is not a tutorial, on purpose. In Diátaxis terms, `sample/` holds the tutorials and how-to guides, and `docs/` holds the explanation and the reference. It still runs, because an explanation that cannot run soon stops matching the code it describes. It runs against `docs/sizing/data/`, its own copy of the sample satellite's input files. The copy is on purpose. The figures quoted below are true only of the data that produced them. If this notebook read `sample/sizing/data/`, a change to the tutorial's inputs would change the figures here without raising an error.

The budget references, `data_budget_ref.ipynb`, `delta_v_ref.ipynb` and `agility_ref.ipynb`, each assume the mission. They point here rather than describe it again.

In [1]:
import os
from pathlib import Path

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from docs/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat import MU_EARTH, R_EARTH, u
from quicksat.utils.mission import Mission

## One mission, stated once

Three budgets need the mission, and each needs something different from it:

| budget | needs |
|---|---|
| data | the period, and how many orbits fit in a day |
| delta-V | the circular velocity, for Hohmann transfers and plane changes; and the design life, which scales the recurring manoeuvres |
| agility | the ground track speed, to turn a slew time into a distance on the ground |

Everything in that table except the design life derives from one number, the altitude. That number could be written into three config files. This would work until someone changed one of them. The budgets would then describe different satellites, and nothing would report it. So the altitude goes in `docs/sizing/data/mission.yaml`, outside any one budget's config. The derivations are in one class, rather than in three copies of Kepler's third law.

In [2]:
mission = Mission.from_yaml_file("docs/sizing/data/mission.yaml")
print(Path("docs/sizing/data/mission.yaml").read_text())
mission

# Shared mission. Read by every module that needs any of it: the data budget for
# period and orbits per day, delta-V for its Hohmann transfers and for scaling the
# recurring manoeuvres, agility for ground track speed.
#
# A fact more than one module can use lives here; a fact only one module can use
# stays in that module's own config -- the Isp, say.
#
# The orbit is circular throughout. quicksat models no eccentricity.

altitude: 500 km
inclination: 97.4 deg      # sun-synchronous at this altitude
duration: 7 yr             # design life, scales the recurring manoeuvres



Mission(altitude=<Quantity 500. km>, inclination=<Quantity 97.4 deg>, duration=<Quantity 7. yr>)

## The file

Three settings. Each is text with a unit, parsed by astropy on load.

| setting | meaning | checked |
|---|---|---|
| `altitude` | height above the Earth's equatorial radius, `R_EARTH` | must be a non-negative length |
| `inclination` | orbit plane inclination | must be a non-negative angle |
| `duration` | design life | must be a non-negative time |

Nothing derives from `inclination`. The delta-V budget's plane change works from the angle that the manoeuvre asks for, not from the plane the spacecraft starts in. So the inclination is loaded and validated, and nothing else uses it. It is in the file because the file describes the orbit, not because anything uses it yet.

Every check is a real one. `500 kg` is rejected on load, instead of becoming a wrong number. So is an inclination with no unit, for the reason given in the units section below.

In [3]:
for text in (
    "altitude: 500 kg\ninclination: 97.4 deg\nduration: 7 yr",
    "altitude: -500 km\ninclination: 97.4 deg\nduration: 7 yr",
    "altitude: 500 km\ninclination: 97.4 km\nduration: 7 yr",
    "altitude: 500 km\ninclination: 97.4\nduration: 7 yr",
    "altitude: 500 km\ninclination: 97.4 deg\nduration: 7 km",
    "altitude: 500 km\ninclination: 97.4 deg\nduration: -7 yr",
    "inclination: 97.4 deg\nduration: 7 yr",
    "altitude: 500 km\ninclination: 97.4 deg",
):
    label = text.replace("\n", ", ")
    try:
        Mission.from_yaml_text(text)
        print(f"{label:56s} accepted")
    except ValueError as exc:
        reason = str(exc).splitlines()[2].strip().split(" [type=")[0]
        print(f"{label:56s} rejected - {reason}")

altitude: 500 kg, inclination: 97.4 deg, duration: 7 yr  rejected - Value error, Value must have length dimensions, got '500.0 kg'
altitude: -500 km, inclination: 97.4 deg, duration: 7 yr rejected - Value error, Length must not be negative, got '-500.0 km'
altitude: 500 km, inclination: 97.4 km, duration: 7 yr   rejected - Value error, Value must have angle dimensions, got '97.4 km'
altitude: 500 km, inclination: 97.4, duration: 7 yr      rejected - Value error, Value must have angle dimensions, got '97.4'
altitude: 500 km, inclination: 97.4 deg, duration: 7 km  rejected - Value error, Value must have time dimensions, got '7.0 km'
altitude: 500 km, inclination: 97.4 deg, duration: -7 yr rejected - Value error, Time must not be negative, got '-7.0 yr'
inclination: 97.4 deg, duration: 7 yr                    rejected - Field required
altitude: 500 km, inclination: 97.4 deg                  rejected - Field required


## How long it flies

`duration` is the design life. It is here for the same reason as the altitude: more than one module can use it. One module uses it today. The delta-V budget multiplies a `recurring` manoeuvre's yearly count by it. So a drag make-up flown four times a year over seven years comes to twenty-eight burns. The data budget could report totals over the life, and the power and radiator work will need it.

It used to be in `delta_v_config.yaml`, under a `mission:` section of its own. By the rule that this file follows, that was the wrong place:

> A fact more than one module can use belongs in the shared file. A fact only one module can use stays in that module's own config.

The Isp stays in the delta-V config, because nothing else can use it. The duration moved here, because other modules can. A budget's config is also a poor place for a value that a second budget will need later. The second budget would most likely get a copy.

**It is required rather than optional.** A file that says where the spacecraft is, but not how long it stays there, describes an orbit, not a mission. If the duration were optional, every module that uses it would have to check for a missing value. That would put the same check in three places, to save one line in one file.

In [4]:
print(f"duration            {mission.duration}")
print(f"  as years          {mission.duration.to('year'):.1f}")
print(f"  as days           {mission.duration.to('day'):.0f}")

# what the delta-V budget does with it: a recurring count is stated per year
per_year = 4
years = mission.duration.to("year").value
print(f"\na manoeuvre flown {per_year}x a year becomes "
      f"{per_year * years:.0f} occurrences over the life")

duration            7.0 yr
  as years          7.0 yr
  as days           2557 d

a manoeuvre flown 4x a year becomes 28 occurrences over the life


## Earth constants

quicksat takes its Earth constants from astropy. `R_EARTH` is `astropy.constants.R_earth`, and `MU_EARTH` is `astropy.constants.GM_earth`. astropy's defaults are the IAU 2015 nominal values, 6378.1 km and 398600.4 km³/s². Standard gravity for the rocket equation comes from the same place, as `astropy.constants.g0`.

Nominal values are conversion constants, not a geodetic model. They differ slightly from the geodetic figures. WGS-84 puts the equatorial radius at 6378.137 km, and EGM96 measures GM as 398600.4418 km³/s². At 500 km, the difference moves the period by 0.05 s and the velocity by 0.02 m/s. The cell below shows this. Neither change matters for sizing.

**`MU_EARTH` is a GM, not $G$ × Earth mass.** $G$ is one of the least precisely known constants in physics, measured to about five significant figures. The product $GM$ is measured directly from spacecraft tracking, to about nine. Working μ out from $G$ and a mass would lose four significant figures. The error would then reach every period and velocity the tool computes. astropy works the same way: it derives its `M_earth` from `GM_earth` and $G$.

In [5]:
import math

# astropy.constants creates its constants at import, so pyright cannot see them
from astropy.constants import G, g0  # pyright: ignore[reportAttributeAccessIssue]

print(f"R_EARTH   {R_EARTH.to('km'):.1f}   ({R_EARTH.reference})")
print(f"MU_EARTH  {MU_EARTH.to('km**3/s**2'):.1f}   ({MU_EARTH.reference})")
print(f"g0        {g0:.5f}")

# against WGS-84's radius and EGM96's measured GM, at the sample altitude
radius = 6378.137 * u.km + mission.altitude
mu = 398600.4418 * u.km**3 / u.s**2
period = (2 * math.pi * (radius**3 / mu) ** 0.5).to("s")
velocity = ((mu / radius) ** 0.5).to("km/s")
print(f"\nperiod    {mission.period:.4f} nominal, {period:.4f} WGS-84 and EGM96"
      f"  ({(mission.period - period).to('s'):+.4f})")
print(f"velocity  {mission.velocity:.6f} nominal, {velocity:.6f} WGS-84 and EGM96"
      f"  ({(mission.velocity - velocity).to('m/s'):+.3f})")

# what deriving mu from G and a mass would cost. G is known to about 2.2e-5
# relative; the IAU Earth mass is itself derived from GM, so this is illustrative
earth_mass = 5.9722e24 * u.kg
derived = (G * earth_mass).to("km**3/s**2")
print(f"\nG x M      {derived:.4f}")
print(f"MU_EARTH   {MU_EARTH.to('km**3/s**2'):.4f}")
print(f"difference {abs(derived - MU_EARTH).to('km**3/s**2'):.4f}"
      f"  ({abs(derived / MU_EARTH - 1).to_value(u.dimensionless_unscaled) * 1e6:.1f} ppm)")

R_EARTH   6378.1 km   (IAU 2015 Resolution B 3)
MU_EARTH  398600.4 km3 / s2   (IAU 2015 Resolution B 3)
g0        9.80665 m / s2

period    5676.9325 s nominal, 5676.9780 s WGS-84 and EGM96  (-0.0455 s)
velocity  7.612628 km / s nominal, 7.612608 km / s WGS-84 and EGM96  (+0.020 m / s)

G x M      398602.5446 km3 / s2
MU_EARTH   398600.4000 km3 / s2
difference 2.1446 km3 / s2  (5.4 ppm)


## What it derives

Circular orbits only. Given $r = R_\\oplus + h$:

$$T = 2\\pi\\sqrt{\\frac{r^3}{\\mu}} \\qquad v = \\sqrt{\\frac{\\mu}{r}} \\qquad v_{\\text{ground}} = v\\,\\frac{R_\\oplus}{r}$$

`orbits_per_day` is dimensionless on purpose, rather than a rate. Its only use is to turn a quantity per orbit into a quantity per day. A dimensionless multiplier does that, and astropy raises no error about orbits not being seconds.

In [6]:
print(f"altitude            {mission.altitude:.0f}")
print(f"radius              {mission.radius:.3f}")
print(f"period              {mission.period:.1f}   ({mission.period.to('min'):.2f})")
print(f"orbits per day      {mission.orbits_per_day:.4f}")
print(f"velocity            {mission.velocity:.4f}")
print(f"ground track speed  {mission.ground_track_speed:.4f}")

# the period and the orbit count are reciprocal by construction
print(f"\nperiod x orbits/day = {(mission.period * mission.orbits_per_day).to('day'):.6f}")

altitude            500 km
radius              6878.100 km
period              5676.9 s   (94.62 min)
orbits per day      15.2195
velocity            7.6126 km / s
ground track speed  7.0592 km / s

period x orbits/day = 1.000000 d


### Ground track speed

The sub-satellite point moves more slowly than the spacecraft, because it traces a smaller circle. The orbit has radius $r$, and the ground track has radius $R_\\oplus$. The speed scales by that ratio.

**Earth rotation is not included.** This is the speed along the ground track over a non-rotating Earth, not the speed relative to a fixed point on the surface. For a near-polar orbit, the two differ by a few hundred metres per second, depending on latitude. That matters for a ground station pass. It does not matter for the agility budget, which uses this speed to find how far the ground track moves during a slew.

In [7]:
ratio = (R_EARTH / mission.radius).to(u.dimensionless_unscaled)
print(f"orbit radius {mission.radius:.0f} vs Earth radius {R_EARTH.to('km'):.0f}")
print(f"ratio {ratio:.4f}, so the ground track runs {(1 - ratio).to('percent'):.1f} slower")
print(f"  orbital     {mission.velocity:.4f}")
print(f"  ground      {mission.ground_track_speed:.4f}")

# what that means for an agility case: distance covered during a 60 s slew
slew = 60 * u.s
print(f"\nduring a {slew:.0f} slew the ground track advances "
      f"{(mission.ground_track_speed * slew).to('km'):.0f}")

orbit radius 6878 km vs Earth radius 6378 km
ratio 0.9273, so the ground track runs 7.3 % slower
  orbital     7.6126 km / s
  ground      7.0592 km / s

during a 60 s slew the ground track advances 424 km


## Units

The altitude may be written in any length unit. It is converted once, on load, and nothing derived from it changes. That is why units are parsed instead of being noted in a comment.

The inclination is checked as strictly as the altitude, because astropy gives angles their own dimension. `AngleQty` rejects `97.4 km`, which is a length. It also rejects a bare `97.4`. A number with no unit would have to be read as radians, which turns a sun-synchronous inclination into 5580.6 degrees. So the file has to say which unit it means.

In [8]:
same = [
    Mission.from_yaml_text(f"altitude: {text}\ninclination: 97.4 deg\nduration: 7 yr")
    for text in ("500 km", "500000 m", "0.5 Mm")
]
for m, text in zip(same, ("500 km", "500000 m", "0.5 Mm"), strict=True):
    print(f"{text:10s} -> altitude {m.altitude.to('km'):.1f}, period {m.period:.1f}")

print("\nan angle has to say what unit it is in, so a bare number is rejected:")
try:
    Mission.from_yaml_text("altitude: 500 km\ninclination: 97.4\nduration: 7 yr")
except ValueError as exc:
    reason = str(exc).splitlines()[2].strip().split(" [type=")[0]
    print(f"  'inclination: 97.4' rejected - {reason}")
print(f"  read as radians, it would have meant {(97.4 * u.rad).to('deg'):.1f}")

500 km     -> altitude 500.0 km, period 5676.9 s
500000 m   -> altitude 500.0 km, period 5676.9 s
0.5 Mm     -> altitude 500.0 km, period 5676.9 s

an angle has to say what unit it is in, so a bare number is rejected:
  'inclination: 97.4' rejected - Value error, Value must have angle dimensions, got '97.4'
  read as radians, it would have meant 5580.6 deg


## Limitations

What the mission model does not do, on purpose:

- **Circular only.** No eccentricity, so no apogee, perigee or true anomaly. A Hohmann transfer in the delta-V budget is computed between two circular orbits rather than along an ellipse.
- **No perturbations.** No J2, so no nodal regression and no argument-of-perigee drift. Sun-synchronicity is only stated in the config file. The tool does not check or maintain it.
- **No drag, and no epoch.** The altitude does not decay, so a mission-end orbit is the same as a mission-start one. There is no date anywhere in the model.
- **Earth only.** `R_EARTH` and `MU_EARTH` are fixed in the derivations, rather than passed in as a central body.
- **A sphere, for the radius.** `R_EARTH` is an *equatorial* radius, so a polar orbit's true altitude above the surface varies by about 21 km over an orbit. This does not matter for sizing. It is wrong for anything related to pointing.
- **Inclination is carried, not used.** It is validated as an angle on load, and nothing derives from it.
- **The duration is a scalar, not a timeline.** It is one number for the whole life. There are no commissioning, operations and disposal phases with their own lengths, and no epoch to place them on.